# Measurment of thermocouple voltage during melting of tin

Import the serial library, scan for port to identify the serial com, define the com address

In [1]:
import serial;

In [2]:
import serial.tools.list_ports as port_list;
ports = list(port_list.comports());
for port in ports: print(port);

COM3 - USB Serial Port (COM3)


In [3]:
serial_address="COM3";
print(serial_address);

COM3


## Preliminary work to test the serial communication with the Keithley2000

Ask for the instrument ID, set the insturment to Volt DC, ask that the instrument to check if it is correctly configured in this mode configured, set the integration time to 5, the range to mV <br>
Retrieve one data

In [4]:
with serial.Serial(serial_address,baudrate=9600,bytesize=8,parity='N',stopbits=1,timeout=.05) as ser:

    ser.write(b'*IDN?\r')
    l=ser.readline()
    response=l.decode('ascii',errors='replace')
    print(response)


KEITHLEY INSTRUMENTS INC.,MODEL 2000,4416105,B01  /A02  



In [5]:
with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=1) as ser:
    ser.write(b":CONF:VOLT:DC\r")
    ser.write(b":SENS:FUNC?\r")
    print(ser.readline().decode('ascii').strip())   # should print "VOLT:DC"

"VOLT:DC"


In [6]:
import time

with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=2) as ser:
    ser.write(b":SENS:FUNC 'VOLT:DC'\r")
    ser.write(b":SENS:VOLT:DC:NPLC 5\r")
    ser.write(b":SENS:VOLT:DC:RANG 0.1\r")
    time.sleep(0.5)

    for cmd in [b":SENS:FUNC?\r", b":SENS:VOLT:DC:NPLC?\r", b":SENS:VOLT:DC:RANG?\r"]:
        ser.reset_input_buffer()
        ser.write(cmd)
        print(cmd.decode().strip(), "->", ser.readline().decode('ascii').strip())

:SENS:FUNC? -> "VOLT:DC"
:SENS:VOLT:DC:NPLC? -> 5.00
:SENS:VOLT:DC:RANG? -> 0.100000


In [7]:
import time

with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=3) as ser:
    ser.reset_input_buffer()
    ser.write(b":READ?\r")
    reading = ser.readline().decode('ascii').strip()
    print("Reading:", reading, "V")

Reading: +3.04687512E-02 V


Transform the data to a float number

In [8]:
value = float(reading)
print(value, "V")
print(value * 1000, "mV")

0.0304687512 V
30.4687512 mV


---

Prepare for the measurments, you will need to retrieve data every 1s and store both time and voltage and temperature in arrays during at least 900s. Convert the data to temperature using the polynomial equation for a K-type thermocouple from https://aviatechno.net/thermo/table_its90.php?type=K

Type Markdown and LaTeX: $\alpha^2$

You may use the following librairies to acquire and plot the data. Save them to a file.

from numpy import *
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from time import sleep

## Thermocouple measurement (type K)

The thermoelectric voltage $x$ (in mV) is converted to temperature with the ITS-90 polynomial:

$$T_{90} = c_0 + c_1 x + c_2 x^2 + \dots + c_n x^n$$

with $T_{90}$ in °C and $x$ in mV. The coefficients depend on the voltage range:

| Range | $-5.891 \to 0$ mV | $0 \to 20.644$ mV | $20.644 \to 54.866$ mV |
|---|---|---|---|
| Temperature | $-200 \to 0$ °C | $0 \to 500$ °C | $500 \to 1372$ °C |

The meter reads the voltage relative to the reference junction, so $T = T_{90} + T_{ref}$.

In [9]:
from numpy import *
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from time import sleep
import time, serial

# ITS-90 type K, voltage (mV) -> temperature (°C), coefficients c0..cn
c_neg  = [0, 2.5173462E+1, -1.1662878E+0, -1.0833638E+0, -8.9773540E-1,
          -3.7342377E-1, -8.6632643E-2, -1.0450598E-2, -5.1920577E-4]      # -5.891 to 0 mV
c_mid  = [0, 2.5083550E+1, 7.8601060E-2, -2.5031310E-1, 8.3152700E-2,
          -1.2280340E-2, 9.8040360E-4, -4.4130300E-5, 1.0577340E-6, -1.0527550E-8]  # 0 to 20.644 mV
c_high = [-1.3180580E+2, 4.8302220E+1, -1.6460310E+0, 5.4647310E-2,
          -9.6507150E-4, 8.8021930E-6, -3.1108100E-8]                      # 20.644 to 54.866 mV

def volt_to_temp(V, T_ref=0.0):
    """V in volts -> temperature in °C. T_ref = reference junction temperature (°C)."""
    x = V * 1000.0   # V -> mV
    if x < 0:
        T = polyval(c_neg[::-1], x)      # polyval wants highest power first
    elif x <= 20.644:
        T = polyval(c_mid[::-1], x)
    else:
        T = polyval(c_high[::-1], x)
    return float(T) + T_ref

print(volt_to_temp(0.0), volt_to_temp(4.096e-3), volt_to_temp(41.276e-3))  # expect ~0, ~100, ~1000 °C

0.0 99.96328562604403 999.9871795182318


In [ ]:
duration = 900      # s
dt = 1.0            # s
T_ref = 0.0         # °C, set to room temperature if the reference junction is not at 0 °C

t_list, V_list, T_list = [], [], []

with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=2) as ser:
    # force the meter into the right mode, regardless of whatever the front panel / a
    # previous session left it in (e.g. it was found on the 10V range, not 100mV)
    ser.write(b":SENS:FUNC 'VOLT:DC'\r")
    ser.write(b":SENS:VOLT:DC:RANG 0.1\r")
    ser.write(b":SENS:VOLT:DC:NPLC 5\r")
    time.sleep(0.5)

    t0 = time.time()
    k = 0
    try:
        while k * dt <= duration:
            wait = t0 + k * dt - time.time()   # note: max() is numpy.max here (from numpy import *),
            if wait > 0:                       # so we branch instead of calling max(0, wait)
                sleep(wait)

            ser.reset_input_buffer()
            ser.write(b":READ?\r")
            time.sleep(0.2)   # give the meter time to answer before we read the line
            raw = ser.readline().decode('ascii', errors='replace').strip()
            t = time.time() - t0

            if raw == "":
                print(f"t={t:6.1f}s: no reply, skipping point")
                k += 1
                continue
            try:
                V = float(raw)
            except ValueError:
                print(f"t={t:6.1f}s: unreadable reply {raw!r}, skipping point")
                k += 1
                continue

            T = nan if abs(V) > 1e30 else volt_to_temp(V, T_ref)   # overrange -> nan
            t_list.append(t); V_list.append(V); T_list.append(T)

            k += 1
    except KeyboardInterrupt:
        print("Stopped manually, data kept.")

t_arr, V_arr, T_arr = array(t_list), array(V_list), array(T_list)

# plot once collection is done
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax1.plot(t_arr, V_arr * 1000, 'b.-'); ax1.set_ylabel("Voltage (mV)")
ax2.plot(t_arr, T_arr, 'r.-');       ax2.set_ylabel("Temperature (°C)")
ax2.set_xlabel("Time (s)")
plt.show()

# save to file
savetxt("thermocouple_data.txt", column_stack((t_arr, V_arr, T_arr)),
        header="time_s  voltage_V  temperature_C", fmt="%.6f")
print(len(t_arr), "points saved to thermocouple_data.txt")